In [1]:
!pip install mup

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# -*- coding: utf-8 -*-
"""
Part 3: µP Scaling and Extrapolation

This script compares Standard Parameterization (SP) from Part 2 vs µP.
It loads SP results from Part 2, performs µP experiments:
  - LR sweep on smallest µP model
  - Train µP models at all sizes
  - Compare scaling laws
  - Extrapolate to 10x largest model
"""

import math
import os
import random
import time
import traceback

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tokenizers import Tokenizer

try:
    import mup
    from mup import MuReadout
    MUP_AVAILABLE = True
except Exception:
    MUP_AVAILABLE = False

# Paths
TOKENIZER_PATH = 'svg_tokenizer_6.json'
TRAIN_TOKENS_PATH = 'train_tokens_6.pt'
VAL_TOKENS_PATH = 'val_tokens_6.pt'
PART2_DIR = 'part2_scaling_results'
OUTPUT_DIR = 'part3_mup_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("PART 3: µP SCALING AND EXTRAPOLATION")
print("="*60)
print(f"Output directory: {OUTPUT_DIR}")
print("Note: This script uses checkpoint/resume logic.")
print("  If interrupted, run again to resume from the last completed stage.")
print("  Stages: 1) LR sweep 2) Train models 3) Fit scaling 4) Plot & analyze")
print("="*60)

for p in [TOKENIZER_PATH, TRAIN_TOKENS_PATH, VAL_TOKENS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing: {p}')

# Model definitions (similar to Part 2, but with µP support)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, mup=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim * n_heads == d_model
        self.mup = mup
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch = query.size(0)
        Q = self.q_linear(query)
        K = self.k_linear(key)
        V = self.v_linear(value)
        Q = Q.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # For µP, use 1/d_model instead of 1/sqrt(head_dim)
        scale = self.d_model if self.mup else math.sqrt(self.head_dim)
        energy = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float('-inf'))
        energy = torch.clamp(energy, min=-50.0, max=50.0)
        attn = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attn), V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(batch, -1, self.d_model)
        return self.fc_out(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, mup=False):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout, mup)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        a = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(a))
        f = self.ff(x)
        x = self.norm2(x + self.dropout(f))
        return x

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=1024, padding_idx=None, mup=False):
        super().__init__()
        self.mup = mup
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.initial_ln = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout, mup) for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d_model)
        if mup:
            self.fc_out = MuReadout(d_model, vocab_size)
        else:
            self.fc_out = nn.Linear(d_model, vocab_size)

    def _generate_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask

    def forward(self, src):
        mask = self._generate_mask(src.size(1), src.device)
        x = self.token_embedding(src)
        x = self.pos_enc(x * math.sqrt(self.token_embedding.embedding_dim))
        x = self.initial_ln(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_ln(x)
        return self.fc_out(x)

class ModelConfig:
    def __init__(self, name, d_model, n_layers, n_heads, d_ff):
        self.name = name
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.d_ff = d_ff

class TokenDataset:
    def __init__(self, sequences):
        self.data = sequences

    def sample_batch(self, batch_size, block_size, device):
        x_batch = []
        y_batch = []
        while len(x_batch) < batch_size:
            seq = random.choice(self.data)
            if len(seq) < block_size + 1:
                continue
            start = random.randint(0, len(seq) - block_size - 1)
            chunk = seq[start:start + block_size + 1]
            x_batch.append(torch.tensor(chunk[:-1], dtype=torch.long))
            y_batch.append(torch.tensor(chunk[1:], dtype=torch.long))
        x = torch.stack(x_batch).to(device)
        y = torch.stack(y_batch).to(device)
        return x, y

def get_scheduler(optimizer, total_steps, warmup_steps):
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=max(1, warmup_steps))
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

def steps_per_epoch(total_tokens, batch_size, block_size):
    tokens_per_step = batch_size * block_size
    return max(1, math.ceil(total_tokens / tokens_per_step))

def train_one_epoch(model, dataset, optimizer, scheduler, device, steps, batch_size, block_size, log_interval=100):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running = 0.0
    recorded = []
    start = time.time()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    for step in range(1, steps + 1):
        x, y = dataset.sample_batch(batch_size, block_size, device)
        out = model(x)
        loss = criterion(out.view(-1, out.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item()
        if step % log_interval == 0 or step == steps:
            interval = min(log_interval, step)
            avg = running / interval
            recorded.append(avg)
            print(f"Step {step}/{steps}, avg_loss={avg:.4f}, lr={scheduler.get_last_lr()[0]:.6g}")
            running = 0.0
    elapsed = time.time() - start
    throughput = steps * batch_size * block_size / elapsed
    peak = None
    if device == 'cuda':
        peak = torch.cuda.max_memory_allocated() / 1024**3
    return recorded, elapsed, throughput, peak

def evaluate(model, dataset, device, steps=100, batch_size=8, block_size=512):
    model.eval()
    crit = nn.CrossEntropyLoss()
    s = 0.0
    with torch.no_grad():
        for _ in range(steps):
            x, y = dataset.sample_batch(batch_size, block_size, device)
            out = model(x)
            loss = crit(out.view(-1, out.size(-1)), y.view(-1))
            s += loss.item()
    return s / steps

def fit_scaling_law(params, losses):
    params = np.asarray(params)
    losses = np.asarray(losses)
    best = None
    for c in np.linspace(0.0, losses.min() * 0.9, 200):
        adjusted = losses - c
        if np.any(adjusted <= 0):
            continue
        lp = np.log(params)
        la = np.log(adjusted)
        coeffs = np.polyfit(lp, la, 1)
        alpha = -coeffs[0]
        a = math.exp(coeffs[1])
        pred = a * params ** (-alpha) + c
        rss = np.sum((pred - losses) ** 2)
        if best is None or rss < best['rss']:
            best = {'a': a, 'alpha': alpha, 'c': c, 'rss': rss}
    if best is None:
        raise RuntimeError('Fit failed')
    return best

def plot_comparison(sp_results, mup_results, fit_sp, fit_mup, outdir):
    plt.figure(figsize=(8, 6))
    sp_params = [r['params'] for r in sp_results]
    sp_losses = [r['val_loss'] for r in sp_results]
    mup_params = [r['params'] for r in mup_results]
    mup_losses = [r['val_loss'] for r in mup_results]
    plt.plot(sp_params, sp_losses, 'o-', label='SP')
    plt.plot(mup_params, mup_losses, 's-', label='µP')
    x_line = np.logspace(math.log10(min(sp_params + mup_params)), math.log10(max(sp_params + mup_params)), 200)
    y_sp = fit_sp['a'] * x_line ** (-fit_sp['alpha']) + fit_sp['c']
    y_mup = fit_mup['a'] * x_line ** (-fit_mup['alpha']) + fit_mup['c']
    plt.plot(x_line, y_sp, '--', color='C0', alpha=0.6, label=f'SP fit α={fit_sp["alpha"]:.3f}')
    plt.plot(x_line, y_mup, '--', color='C1', alpha=0.6, label=f'µP fit α={fit_mup["alpha"]:.3f}')
    plt.xscale('log')
    plt.xlabel('Parameters (M)')
    plt.ylabel('Validation Loss')
    plt.title('SP vs µP Scaling Laws')
    plt.grid(True, which='both', ls='--', alpha=0.4)
    plt.legend()
    path = os.path.join(outdir, 'sp_vs_mup_scaling.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    return path

def apply_mup(model, base_model):
    if not MUP_AVAILABLE:
        raise ImportError("Install mup: pip install mup")
    mup.set_base_shapes(model, base_model)
    # Initialize weights as per µP
    for module in model.modules():
        if isinstance(module, nn.Linear):
            mup.init.kaiming_normal_(module.weight, mode='fan_in')
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0)
        elif isinstance(module, MuReadout):
            # MuReadout handles its own init
            pass
    return model

def extrapolate(fit, max_trained_param):
    target = max_trained_param * 10.0
    pred = fit['a'] * target ** (-fit['alpha']) + fit['c']
    uncertainty = math.sqrt(fit['rss']) / max(1.0, len(fit) - 2)  # rough
    return target, pred, uncertainty

def list_checkpoints(output_dir):
    """List all available checkpoints."""
    checkpoints = {
        'lr_sweep': os.path.exists(os.path.join(output_dir, 'lr_results.pt')),
        'model_training': os.path.exists(os.path.join(output_dir, 'mup_results.pt')),
        'scaling_fit': os.path.exists(os.path.join(output_dir, 'fit_mup.pt')),
        'plots': os.path.exists(os.path.join(output_dir, 'sp_vs_mup_scaling.png')),
    }
    return checkpoints

def clear_checkpoints(output_dir, stages=['lr_sweep', 'model_training', 'scaling_fit']):
    """Clear specific checkpoints to restart from scratch or specific stage.
    stages: list of ['lr_sweep', 'model_training', 'scaling_fit']
    """
    files_to_remove = {
        'lr_sweep': ['lr_results.pt', 'best_lr.txt'],
        'model_training': ['mup_results.pt'],
        'scaling_fit': ['fit_mup.pt', 'part3_results.pt'],
    }
    for stage in stages:
        for filename in files_to_remove.get(stage, []):
            path = os.path.join(output_dir, filename)
            if os.path.exists(path):
                os.remove(path)
                print(f"Removed {path}")
    print(f"Cleared checkpoints for stages: {stages}")

# Load SP results from Part 2
sp_results = []
for name in ['tiny', 'small', 'medium', 'large', 'xl']:
    path = os.path.join(PART2_DIR, f"{name}.pt")
    if os.path.exists(path):
        sp_results.append(torch.load(path))
    else:
        print(f"Warning: {path} not found")

if not sp_results:
    raise FileNotFoundError("No SP results from Part 2 found. Run Part 2 first.")

fit_sp = fit_scaling_law([r['params'] for r in sp_results], [r['val_loss'] for r in sp_results])

# Load data
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
train_tokens = torch.load(TRAIN_TOKENS_PATH)
val_tokens = torch.load(VAL_TOKENS_PATH)
total_train = sum(len(s) for s in train_tokens)

vocab_size = tokenizer.get_vocab_size()
pad_id = None

block_size = 512
tokens_per_batch = 4096
batch_size = max(32, tokens_per_batch // block_size)
epoch_steps = steps_per_epoch(total_train, batch_size, block_size)
warmup_steps = max(1, epoch_steps // 5)

train_ds = TokenDataset(train_tokens)
val_ds = TokenDataset(val_tokens)

model_sizes = [
    ModelConfig('tiny', 128, 4, 4, 512),
    ModelConfig('small', 192, 6, 6, 768),
    ModelConfig('medium', 384, 6, 6, 1536),
    ModelConfig('large', 512, 8, 8, 2048),
    ModelConfig('xl', 768, 12, 12, 3072),
]

# Checkpointing paths
lr_results_path = os.path.join(OUTPUT_DIR, 'lr_results.pt')
mup_results_path = os.path.join(OUTPUT_DIR, 'mup_results.pt')
fit_mup_path = os.path.join(OUTPUT_DIR, 'fit_mup.pt')
best_lr_path = os.path.join(OUTPUT_DIR, 'best_lr.txt')
checkpoint_meta_path = os.path.join(OUTPUT_DIR, 'checkpoint_meta.pt')

# LR sweep for µP tiny
if not MUP_AVAILABLE:
    print("mup not available, install pip install mup")
else:
    # ============ STAGE 1: LR SWEEP ============
    if os.path.exists(lr_results_path) and os.path.exists(best_lr_path):
        print("Resuming from checkpoint: Loading LR sweep results...")
        lr_results = torch.load(lr_results_path)
        with open(best_lr_path, 'r') as f:
            best_lr = float(f.read().strip())
        print(f"Loaded best µP LR: {best_lr}")
    else:
        print("Running µP LR sweep...")
        tiny_cfg = model_sizes[0]
        base_model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
        lr_candidates = np.logspace(-5, -3, 7)
        best_lr = None
        best_val = float('inf')
        lr_results = []
        for i, lr in enumerate(lr_candidates):
            print(f"  LR sweep: {i+1}/{len(lr_candidates)}, lr={lr:.6f}")
            model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
            model = apply_mup(model, base_model)
            model = model.to(device)
            optimizer = optim.AdamW(model.parameters(), lr=lr)
            scheduler = get_scheduler(optimizer, max(1, min(epoch_steps, 800)), max(1, warmup_steps//4))
            train_one_epoch(model, train_ds, optimizer, scheduler, device, min(epoch_steps, 800), batch_size, block_size)
            val_loss = evaluate(model, val_ds, device)
            lr_results.append({'lr': lr, 'val_loss': val_loss})
            if val_loss < best_val:
                best_val = val_loss
                best_lr = lr
            print(f"    val_loss={val_loss:.4f}, best_lr={best_lr:.6f}")
        
        # Checkpoint LR sweep
        torch.save(lr_results, lr_results_path)
        with open(best_lr_path, 'w') as f:
            f.write(str(best_lr))
        print(f"Checkpointed LR sweep. Best µP LR: {best_lr}")

    # ============ STAGE 2: TRAIN ALL µP MODELS ============
    if os.path.exists(mup_results_path):
        print("Resuming from checkpoint: Loading µP training results...")
        mup_results = torch.load(mup_results_path)
        print(f"Loaded {len(mup_results)} µP model results")
    else:
        print("Training all µP models...")
        tiny_cfg = model_sizes[0]
        base_model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
        
        mup_results = []
        for i, cfg in enumerate(model_sizes):
            print(f"  Training µP {cfg.name} ({i+1}/{len(model_sizes)})...")
            model = DecoderOnlyTransformer(vocab_size, cfg.d_model, cfg.n_layers, cfg.n_heads, cfg.d_ff, mup=True)
            model = apply_mup(model, base_model)
            model = model.to(device)
            param_count = sum(p.numel() for p in model.parameters()) / 1e6
            optimizer = optim.AdamW(model.parameters(), lr=best_lr)
            scheduler = get_scheduler(optimizer, epoch_steps, warmup_steps)
            train_losses, elapsed, throughput, peak = train_one_epoch(model, train_ds, optimizer, scheduler, device, epoch_steps, batch_size, block_size)
            val_loss = evaluate(model, val_ds, device)
            mup_results.append({'name': cfg.name, 'params': param_count, 'val_loss': val_loss, 'train_curve': train_losses})
            print(f"    µP {cfg.name}: {param_count:.2f}M, val_loss={val_loss:.4f}, elapsed={elapsed:.1f}s")
            
            # Checkpoint after each model
            torch.save(mup_results, mup_results_path)
        
        print("Checkpointed all µP training results")

    # ============ STAGE 3: FIT SCALING LAW ============
    if os.path.exists(fit_mup_path):
        print("Resuming from checkpoint: Loading µP scaling law fit...")
        fit_mup = torch.load(fit_mup_path)
        print(f"Loaded µP fit: α={fit_mup['alpha']:.4f}, a={fit_mup['a']:.6f}, c={fit_mup['c']:.6f}")
    else:
        print("Fitting µP scaling law...")
        fit_mup = fit_scaling_law([r['params'] for r in mup_results], [r['val_loss'] for r in mup_results])
        torch.save(fit_mup, fit_mup_path)
        print(f"Checkpointed µP scaling law: α={fit_mup['alpha']:.4f}, a={fit_mup['a']:.6f}, c={fit_mup['c']:.6f}")

    # ============ STAGE 4: PLOTTING AND ANALYSIS ============
    print("\nGenerating comparison plot...")
    plot_comparison(sp_results, mup_results, fit_sp, fit_mup, OUTPUT_DIR)
    print(f"Plot saved to {OUTPUT_DIR}/sp_vs_mup_scaling.png")

    # Extrapolate
    print("\nExtrapolating to 10x model size...")
    max_param = max(r['params'] for r in mup_results)
    target, pred, unc = extrapolate(fit_mup, max_param)
    print(f"Extrapolated at {target:.2f}M params: loss={pred:.6f} ± {unc:.6f}")

    # Final summary and save
    print("\n" + "="*60)
    print("PART 3 SUMMARY")
    print("="*60)
    print(f"SP scaling exponent α: {fit_sp['alpha']:.4f}")
    print(f"µP scaling exponent α: {fit_mup['alpha']:.4f}")
    print(f"SP results: {len(sp_results)} models, {min(r['params'] for r in sp_results):.1f}M - {max(r['params'] for r in sp_results):.1f}M params")
    print(f"µP results: {len(mup_results)} models, {min(r['params'] for r in mup_results):.1f}M - {max(r['params'] for r in mup_results):.1f}M params")
    print(f"µP extrapolated loss at {target:.2f}M params: {pred:.6f}")
    print("="*60)

    # Save all results
    final_results = {
        'sp_results': sp_results, 
        'mup_results': mup_results, 
        'fit_sp': fit_sp, 
        'fit_mup': fit_mup,
        'best_lr': best_lr,
        'lr_results': lr_results,
        'extrapolation': {'target': target, 'pred': pred, 'unc': unc}
    }
    torch.save(final_results, os.path.join(OUTPUT_DIR, 'part3_results.pt'))
    print(f"\nAll results saved to {OUTPUT_DIR}/part3_results.pt")

PART 3: µP SCALING AND EXTRAPOLATION
Output directory: part3_mup_results
Note: This script uses checkpoint/resume logic.
  If interrupted, run again to resume from the last completed stage.
  Stages: 1) LR sweep 2) Train models 3) Fit scaling 4) Plot & analyze
Running µP LR sweep...
  LR sweep: 1/7, lr=0.000010
Step 100/800, avg_loss=9.4784, lr=2.7248e-06
Step 200/800, avg_loss=8.5420, lr=5.4496e-06
Step 300/800, avg_loss=7.2140, lr=8.17439e-06
Step 400/800, avg_loss=6.2805, lr=9.85737e-06
Step 500/800, avg_loss=5.7331, lr=7.84721e-06
Step 600/800, avg_loss=5.4066, lr=4.40286e-06
Step 700/800, avg_loss=5.2100, lr=1.2593e-06
Step 800/800, avg_loss=5.1474, lr=0
    val_loss=4.8876, best_lr=0.000010
  LR sweep: 2/7, lr=0.000022
Step 100/800, avg_loss=9.2885, lr=5.87041e-06
Step 200/800, avg_loss=7.6203, lr=1.17408e-05
Step 300/800, avg_loss=6.4062, lr=1.76112e-05
Step 400/800, avg_loss=5.5619, lr=2.12371e-05
Step 500/800, avg_loss=4.5968, lr=1.69063e-05
Step 600/800, avg_loss=3.8721, lr=9

AssertionError: `base_shapes` has extra names set(). `shapes` has extra names {'layers.4.ff.0.bias', 'layers.5.norm1.bias', 'layers.4.ff.2.bias', 'layers.4.attention.k_linear.bias', 'layers.4.norm1.weight', 'layers.4.attention.fc_out.weight', 'layers.5.ff.0.bias', 'layers.4.ff.0.weight', 'layers.4.attention.v_linear.weight', 'layers.5.norm1.weight', 'layers.5.attention.fc_out.bias', 'layers.4.attention.v_linear.bias', 'layers.5.attention.k_linear.bias', 'layers.5.attention.v_linear.weight', 'layers.4.attention.q_linear.weight', 'layers.5.attention.k_linear.weight', 'layers.4.attention.q_linear.bias', 'layers.5.norm2.bias', 'layers.5.ff.0.weight', 'layers.4.norm1.bias', 'layers.4.norm2.weight', 'layers.5.ff.2.bias', 'layers.4.attention.k_linear.weight', 'layers.4.attention.fc_out.bias', 'layers.5.attention.q_linear.bias', 'layers.5.ff.2.weight', 'layers.5.attention.q_linear.weight', 'layers.5.attention.fc_out.weight', 'layers.5.attention.v_linear.bias', 'layers.4.norm2.bias', 'layers.4.

In [ ]:
# -*- coding: utf-8 -*-
"""
Part 3: µP Scaling and Extrapolation

This script compares Standard Parameterization (SP) from Part 2 vs µP.
It loads SP results from Part 2, performs µP experiments:
  - LR sweep on smallest µP model
  - Train µP models at all sizes
  - Compare scaling laws
  - Extrapolate to 10x largest model
"""

import math
import os
import random
import time
import traceback

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tokenizers import Tokenizer

try:
    import mup
    from mup import MuReadout
    MUP_AVAILABLE = True
except Exception:
    MUP_AVAILABLE = False

# Paths
TOKENIZER_PATH = 'svg_tokenizer_6.json'
TRAIN_TOKENS_PATH = 'train_tokens_6.pt'
VAL_TOKENS_PATH = 'val_tokens_6.pt'
PART2_DIR = 'part2_scaling_results'
OUTPUT_DIR = 'part3_mup_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("PART 3: µP SCALING AND EXTRAPOLATION")
print("="*60)
print(f"Output directory: {OUTPUT_DIR}")
print("Note: This script uses checkpoint/resume logic.")
print("  If interrupted, run again to resume from the last completed stage.")
print("  Stages: 1) LR sweep 2) Train models 3) Fit scaling 4) Plot & analyze")
print("="*60)

for p in [TOKENIZER_PATH, TRAIN_TOKENS_PATH, VAL_TOKENS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing: {p}')

# Model definitions (similar to Part 2, but with µP support)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, mup=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim * n_heads == d_model
        self.mup = mup
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch = query.size(0)
        Q = self.q_linear(query)
        K = self.k_linear(key)
        V = self.v_linear(value)
        Q = Q.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # For µP, use 1/d_model instead of 1/sqrt(head_dim)
        scale = self.d_model if self.mup else math.sqrt(self.head_dim)
        energy = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float('-inf'))
        energy = torch.clamp(energy, min=-50.0, max=50.0)
        attn = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attn), V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(batch, -1, self.d_model)
        return self.fc_out(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, mup=False):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout, mup)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        a = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(a))
        f = self.ff(x)
        x = self.norm2(x + self.dropout(f))
        return x

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=1024, padding_idx=None, mup=False):
        super().__init__()
        self.mup = mup
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.initial_ln = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout, mup) for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d_model)
        if mup:
            self.fc_out = MuReadout(d_model, vocab_size)
        else:
            self.fc_out = nn.Linear(d_model, vocab_size)

    def _generate_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask

    def forward(self, src):
        mask = self._generate_mask(src.size(1), src.device)
        x = self.token_embedding(src)
        x = self.pos_enc(x * math.sqrt(self.token_embedding.embedding_dim))
        x = self.initial_ln(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_ln(x)
        return self.fc_out(x)

class ModelConfig:
    def __init__(self, name, d_model, n_layers, n_heads, d_ff):
        self.name = name
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.d_ff = d_ff

class TokenDataset:
    def __init__(self, sequences):
        self.data = sequences

    def sample_batch(self, batch_size, block_size, device):
        x_batch = []
        y_batch = []
        while len(x_batch) < batch_size:
            seq = random.choice(self.data)
            if len(seq) < block_size + 1:
                continue
            start = random.randint(0, len(seq) - block_size - 1)
            chunk = seq[start:start + block_size + 1]
            x_batch.append(torch.tensor(chunk[:-1], dtype=torch.long))
            y_batch.append(torch.tensor(chunk[1:], dtype=torch.long))
        x = torch.stack(x_batch).to(device)
        y = torch.stack(y_batch).to(device)
        return x, y

def get_scheduler(optimizer, total_steps, warmup_steps):
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=max(1, warmup_steps))
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

def steps_per_epoch(total_tokens, batch_size, block_size):
    tokens_per_step = batch_size * block_size
    return max(1, math.ceil(total_tokens / tokens_per_step))

def train_one_epoch(model, dataset, optimizer, scheduler, device, steps, batch_size, block_size, log_interval=100):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running = 0.0
    recorded = []
    start = time.time()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    for step in range(1, steps + 1):
        x, y = dataset.sample_batch(batch_size, block_size, device)
        out = model(x)
        loss = criterion(out.view(-1, out.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item()
        if step % log_interval == 0 or step == steps:
            interval = min(log_interval, step)
            avg = running / interval
            recorded.append(avg)
            print(f"Step {step}/{steps}, avg_loss={avg:.4f}, lr={scheduler.get_last_lr()[0]:.6g}")
            running = 0.0
    elapsed = time.time() - start
    throughput = steps * batch_size * block_size / elapsed
    peak = None
    if device == 'cuda':
        peak = torch.cuda.max_memory_allocated() / 1024**3
    return recorded, elapsed, throughput, peak

def evaluate(model, dataset, device, steps=100, batch_size=8, block_size=512):
    model.eval()
    crit = nn.CrossEntropyLoss()
    s = 0.0
    with torch.no_grad():
        for _ in range(steps):
            x, y = dataset.sample_batch(batch_size, block_size, device)
            out = model(x)
            loss = crit(out.view(-1, out.size(-1)), y.view(-1))
            s += loss.item()
    return s / steps

def fit_scaling_law(params, losses):
    params = np.asarray(params)
    losses = np.asarray(losses)
    best = None
    for c in np.linspace(0.0, losses.min() * 0.9, 200):
        adjusted = losses - c
        if np.any(adjusted <= 0):
            continue
        lp = np.log(params)
        la = np.log(adjusted)
        coeffs = np.polyfit(lp, la, 1)
        alpha = -coeffs[0]
        a = math.exp(coeffs[1])
        pred = a * params ** (-alpha) + c
        rss = np.sum((pred - losses) ** 2)
        if best is None or rss < best['rss']:
            best = {'a': a, 'alpha': alpha, 'c': c, 'rss': rss}
    if best is None:
        raise RuntimeError('Fit failed')
    return best

def plot_comparison(sp_results, mup_results, fit_sp, fit_mup, outdir):
    plt.figure(figsize=(8, 6))
    sp_params = [r['params'] for r in sp_results]
    sp_losses = [r['val_loss'] for r in sp_results]
    mup_params = [r['params'] for r in mup_results]
    mup_losses = [r['val_loss'] for r in mup_results]
    plt.plot(sp_params, sp_losses, 'o-', label='SP')
    plt.plot(mup_params, mup_losses, 's-', label='µP')
    x_line = np.logspace(math.log10(min(sp_params + mup_params)), math.log10(max(sp_params + mup_params)), 200)
    y_sp = fit_sp['a'] * x_line ** (-fit_sp['alpha']) + fit_sp['c']
    y_mup = fit_mup['a'] * x_line ** (-fit_mup['alpha']) + fit_mup['c']
    plt.plot(x_line, y_sp, '--', color='C0', alpha=0.6, label=f'SP fit α={fit_sp["alpha"]:.3f}')
    plt.plot(x_line, y_mup, '--', color='C1', alpha=0.6, label=f'µP fit α={fit_mup["alpha"]:.3f}')
    plt.xscale('log')
    plt.xlabel('Parameters (M)')
    plt.ylabel('Validation Loss')
    plt.title('SP vs µP Scaling Laws')
    plt.grid(True, which='both', ls='--', alpha=0.4)
    plt.legend()
    path = os.path.join(outdir, 'sp_vs_mup_scaling.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    return path

def apply_mup(model, base_model):
    if not MUP_AVAILABLE:
        raise ImportError("Install mup: pip install mup")
    mup.set_base_shapes(model, base_model)
    # Initialize weights as per µP
    for module in model.modules():
        if isinstance(module, nn.Linear):
            mup.init.kaiming_normal_(module.weight, mode='fan_in')
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0)
        elif isinstance(module, MuReadout):
            # MuReadout handles its own init
            pass
    return model

def extrapolate(fit, max_trained_param):
    target = max_trained_param * 10.0
    pred = fit['a'] * target ** (-fit['alpha']) + fit['c']
    uncertainty = math.sqrt(fit['rss']) / max(1.0, len(fit) - 2)  # rough
    return target, pred, uncertainty

def list_checkpoints(output_dir):
    """List all available checkpoints."""
    checkpoints = {
        'lr_sweep': os.path.exists(os.path.join(output_dir, 'lr_results.pt')),
        'model_training': os.path.exists(os.path.join(output_dir, 'mup_results.pt')),
        'scaling_fit': os.path.exists(os.path.join(output_dir, 'fit_mup.pt')),
        'plots': os.path.exists(os.path.join(output_dir, 'sp_vs_mup_scaling.png')),
    }
    return checkpoints

def clear_checkpoints(output_dir, stages=['lr_sweep', 'model_training', 'scaling_fit']):
    """Clear specific checkpoints to restart from scratch or specific stage.
    stages: list of ['lr_sweep', 'model_training', 'scaling_fit']
    """
    files_to_remove = {
        'lr_sweep': ['lr_results.pt', 'best_lr.txt'],
        'model_training': ['mup_results.pt'],
        'scaling_fit': ['fit_mup.pt', 'part3_results.pt'],
    }
    for stage in stages:
        for filename in files_to_remove.get(stage, []):
            path = os.path.join(output_dir, filename)
            if os.path.exists(path):
                os.remove(path)
                print(f"Removed {path}")
    print(f"Cleared checkpoints for stages: {stages}")

# Load SP results from Part 2
sp_results = []
for name in ['tiny', 'small', 'medium', 'large', 'xl']:
    path = os.path.join(PART2_DIR, f"{name}.pt")
    if os.path.exists(path):
        sp_results.append(torch.load(path))
    else:
        print(f"Warning: {path} not found")

if not sp_results:
    raise FileNotFoundError("No SP results from Part 2 found. Run Part 2 first.")

fit_sp = fit_scaling_law([r['params'] for r in sp_results], [r['val_loss'] for r in sp_results])

# Load data
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
train_tokens = torch.load(TRAIN_TOKENS_PATH)
val_tokens = torch.load(VAL_TOKENS_PATH)
total_train = sum(len(s) for s in train_tokens)

vocab_size = tokenizer.get_vocab_size()
pad_id = None

block_size = 512
tokens_per_batch = 4096
batch_size = max(32, tokens_per_batch // block_size)
epoch_steps = steps_per_epoch(total_train, batch_size, block_size)
warmup_steps = max(1, epoch_steps // 5)

train_ds = TokenDataset(train_tokens)
val_ds = TokenDataset(val_tokens)

model_sizes = [
    ModelConfig('tiny',   128, 6, 4, 512),   # 128/4 = 32
    ModelConfig('small',  192, 6, 6, 768),   # 192/6 = 32
    ModelConfig('medium', 384, 6, 12, 1536), # 384/12 = 32
    ModelConfig('large',  512, 6, 16, 2048), # 512/16 = 32
    ModelConfig('xl',     768, 6, 24, 3072), # 768/24 = 32
]

# Checkpointing paths
lr_results_path = os.path.join(OUTPUT_DIR, 'lr_results.pt')
mup_results_path = os.path.join(OUTPUT_DIR, 'mup_results.pt')
fit_mup_path = os.path.join(OUTPUT_DIR, 'fit_mup.pt')
best_lr_path = os.path.join(OUTPUT_DIR, 'best_lr.txt')
checkpoint_meta_path = os.path.join(OUTPUT_DIR, 'checkpoint_meta.pt')

# LR sweep for µP tiny
if not MUP_AVAILABLE:
    print("mup not available, install pip install mup")
else:
    # ============ STAGE 1: LR SWEEP ============
    if os.path.exists(lr_results_path) and os.path.exists(best_lr_path):
        print("Resuming from checkpoint: Loading LR sweep results...")
        lr_results = torch.load(lr_results_path)
        with open(best_lr_path, 'r') as f:
            best_lr = float(f.read().strip())
        print(f"Loaded best µP LR: {best_lr}")
    else:
        print("Running µP LR sweep...")
        tiny_cfg = model_sizes[0]
        base_model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
        lr_candidates = np.logspace(-5, -3, 7)
        best_lr = None
        best_val = float('inf')
        lr_results = []
        for i, lr in enumerate(lr_candidates):
            print(f"  LR sweep: {i+1}/{len(lr_candidates)}, lr={lr:.6f}")
            model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
            model = apply_mup(model, base_model)
            model = model.to(device)
            optimizer = optim.AdamW(model.parameters(), lr=lr)
            scheduler = get_scheduler(optimizer, max(1, min(epoch_steps, 800)), max(1, warmup_steps//4))
            train_one_epoch(model, train_ds, optimizer, scheduler, device, min(epoch_steps, 800), batch_size, block_size)
            val_loss = evaluate(model, val_ds, device)
            lr_results.append({'lr': lr, 'val_loss': val_loss})
            if val_loss < best_val:
                best_val = val_loss
                best_lr = lr
            print(f"    val_loss={val_loss:.4f}, best_lr={best_lr:.6f}")
        
        # Checkpoint LR sweep
        torch.save(lr_results, lr_results_path)
        with open(best_lr_path, 'w') as f:
            f.write(str(best_lr))
        print(f"Checkpointed LR sweep. Best µP LR: {best_lr}")

    # ============ STAGE 2: TRAIN ALL µP MODELS ============
    if os.path.exists(mup_results_path):
        print("Resuming from checkpoint: Loading µP training results...")
        mup_results = torch.load(mup_results_path)
        print(f"Loaded {len(mup_results)} µP model results")
    else:
        print("Training all µP models...")
        tiny_cfg = model_sizes[0]
        base_model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
        
        mup_results = []
        for i, cfg in enumerate(model_sizes):
            print(f"  Training µP {cfg.name} ({i+1}/{len(model_sizes)})...")
            model = DecoderOnlyTransformer(vocab_size, cfg.d_model, cfg.n_layers, cfg.n_heads, cfg.d_ff, mup=True)
            model = apply_mup(model, base_model)
            model = model.to(device)
            param_count = sum(p.numel() for p in model.parameters()) / 1e6
            optimizer = optim.AdamW(model.parameters(), lr=best_lr)
            scheduler = get_scheduler(optimizer, epoch_steps, warmup_steps)
            train_losses, elapsed, throughput, peak = train_one_epoch(model, train_ds, optimizer, scheduler, device, epoch_steps, batch_size, block_size)
            val_loss = evaluate(model, val_ds, device)
            mup_results.append({'name': cfg.name, 'params': param_count, 'val_loss': val_loss, 'train_curve': train_losses})
            print(f"    µP {cfg.name}: {param_count:.2f}M, val_loss={val_loss:.4f}, elapsed={elapsed:.1f}s")
            
            # Checkpoint after each model
            torch.save(mup_results, mup_results_path)
        
        print("Checkpointed all µP training results")

    # ============ STAGE 3: FIT SCALING LAW ============
    if os.path.exists(fit_mup_path):
        print("Resuming from checkpoint: Loading µP scaling law fit...")
        fit_mup = torch.load(fit_mup_path)
        print(f"Loaded µP fit: α={fit_mup['alpha']:.4f}, a={fit_mup['a']:.6f}, c={fit_mup['c']:.6f}")
    else:
        print("Fitting µP scaling law...")
        fit_mup = fit_scaling_law([r['params'] for r in mup_results], [r['val_loss'] for r in mup_results])
        torch.save(fit_mup, fit_mup_path)
        print(f"Checkpointed µP scaling law: α={fit_mup['alpha']:.4f}, a={fit_mup['a']:.6f}, c={fit_mup['c']:.6f}")

    # ============ STAGE 4: PLOTTING AND ANALYSIS ============
    print("\nGenerating comparison plot...")
    plot_comparison(sp_results, mup_results, fit_sp, fit_mup, OUTPUT_DIR)
    print(f"Plot saved to {OUTPUT_DIR}/sp_vs_mup_scaling.png")

    # Extrapolate
    print("\nExtrapolating to 10x model size...")
    max_param = max(r['params'] for r in mup_results)
    target, pred, unc = extrapolate(fit_mup, max_param)
    print(f"Extrapolated at {target:.2f}M params: loss={pred:.6f} ± {unc:.6f}")

    # Final summary and save
    print("\n" + "="*60)
    print("PART 3 SUMMARY")
    print("="*60)
    print(f"SP scaling exponent α: {fit_sp['alpha']:.4f}")
    print(f"µP scaling exponent α: {fit_mup['alpha']:.4f}")
    print(f"SP results: {len(sp_results)} models, {min(r['params'] for r in sp_results):.1f}M - {max(r['params'] for r in sp_results):.1f}M params")
    print(f"µP results: {len(mup_results)} models, {min(r['params'] for r in mup_results):.1f}M - {max(r['params'] for r in mup_results):.1f}M params")
    print(f"µP extrapolated loss at {target:.2f}M params: {pred:.6f}")
    print("="*60)

    # Save all results
    final_results = {
        'sp_results': sp_results, 
        'mup_results': mup_results, 
        'fit_sp': fit_sp, 
        'fit_mup': fit_mup,
        'best_lr': best_lr,
        'lr_results': lr_results,
        'extrapolation': {'target': target, 'pred': pred, 'unc': unc}
    }
    torch.save(final_results, os.path.join(OUTPUT_DIR, 'part3_results.pt'))
    print(f"\nAll results saved to {OUTPUT_DIR}/part3_results.pt")

PART 3: µP SCALING AND EXTRAPOLATION
Output directory: part3_mup_results
Note: This script uses checkpoint/resume logic.
  If interrupted, run again to resume from the last completed stage.
  Stages: 1) LR sweep 2) Train models 3) Fit scaling 4) Plot & analyze
Running µP LR sweep...
  LR sweep: 1/7, lr=0.000010
Step 100/800, avg_loss=9.5924, lr=2.7248e-06
Step 200/800, avg_loss=8.2396, lr=5.4496e-06
Step 300/800, avg_loss=6.9362, lr=8.17439e-06


/home/av4008/.local/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Step 400/800, avg_loss=6.2215, lr=9.85737e-06
Step 500/800, avg_loss=5.8164, lr=7.84721e-06
Step 600/800, avg_loss=5.6015, lr=4.40286e-06
Step 700/800, avg_loss=5.4980, lr=1.2593e-06
Step 800/800, avg_loss=5.4656, lr=0
    val_loss=5.2902, best_lr=0.000010
  LR sweep: 2/7, lr=0.000022
Step 100/800, avg_loss=8.7997, lr=5.87041e-06
Step 200/800, avg_loss=7.1593, lr=1.17408e-05
Step 300/800, avg_loss=6.2413, lr=1.76112e-05
Step 400/800, avg_loss=5.6523, lr=2.12371e-05
Step 500/800, avg_loss=5.1465, lr=1.69063e-05
Step 600/800, avg_loss=4.8022, lr=9.48567e-06
Step 700/800, avg_loss=4.6305, lr=2.71307e-06
Step 800/800, avg_loss=4.5855, lr=0
    val_loss=4.4603, best_lr=0.000022
  LR sweep: 3/7, lr=0.000046
Step 100/800, avg_loss=8.6925, lr=1.26474e-05
Step 200/800, avg_loss=6.7409, lr=2.52948e-05
Step 300/800, avg_loss=5.7650, lr=3.79422e-05
Step 400/800, avg_loss=4.9156, lr=4.57539e-05
Step 500/800, avg_loss=4.1270, lr=3.64235e-05
Step 600/800, avg_loss=3.3957, lr=2.04363e-05
Step 700/800,

In [1]:
# -*- coding: utf-8 -*-
"""
Part 3: µP Scaling and Extrapolation

This script compares Standard Parameterization (SP) from Part 2 vs µP.
It loads SP results from Part 2, performs µP experiments:
  - LR sweep on smallest µP model
  - Train µP models at all sizes
  - Compare scaling laws
  - Extrapolate to 10x largest model
"""

import math
import os
import random
import time
import traceback

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tokenizers import Tokenizer

try:
    import mup
    from mup import MuReadout
    MUP_AVAILABLE = True
except Exception:
    MUP_AVAILABLE = False

# Paths
TOKENIZER_PATH = 'svg_tokenizer_6.json'
TRAIN_TOKENS_PATH = 'train_tokens_6.pt'
VAL_TOKENS_PATH = 'val_tokens_6.pt'
PART2_DIR = 'part2_scaling_results'
OUTPUT_DIR = 'part3_mup_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("PART 3: µP SCALING AND EXTRAPOLATION")
print("="*60)
print(f"Output directory: {OUTPUT_DIR}")
print("Note: This script uses checkpoint/resume logic.")
print("  If interrupted, run again to resume from the last completed stage.")
print("  Stages: 1) LR sweep 2) Train models 3) Fit scaling 4) Plot & analyze")
print("="*60)

for p in [TOKENIZER_PATH, TRAIN_TOKENS_PATH, VAL_TOKENS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing: {p}')

# Model definitions (similar to Part 2, but with µP support)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, mup=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim * n_heads == d_model
        self.mup = mup
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch = query.size(0)
        Q = self.q_linear(query)
        K = self.k_linear(key)
        V = self.v_linear(value)
        Q = Q.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # For µP, use 1/d_model instead of 1/sqrt(head_dim)
        scale = self.d_model if self.mup else math.sqrt(self.head_dim)
        energy = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float('-inf'))
        energy = torch.clamp(energy, min=-50.0, max=50.0)
        attn = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attn), V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(batch, -1, self.d_model)
        return self.fc_out(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, mup=False):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout, mup)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        a = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(a))
        f = self.ff(x)
        x = self.norm2(x + self.dropout(f))
        return x

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=1024, padding_idx=None, mup=False):
        super().__init__()
        self.mup = mup
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.initial_ln = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout, mup) for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d_model)
        if mup:
            self.fc_out = MuReadout(d_model, vocab_size)
        else:
            self.fc_out = nn.Linear(d_model, vocab_size)

    def _generate_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask

    def forward(self, src):
        mask = self._generate_mask(src.size(1), src.device)
        x = self.token_embedding(src)
        x = self.pos_enc(x * math.sqrt(self.token_embedding.embedding_dim))
        x = self.initial_ln(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_ln(x)
        return self.fc_out(x)

class ModelConfig:
    def __init__(self, name, d_model, n_layers, n_heads, d_ff):
        self.name = name
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.d_ff = d_ff

class TokenDataset:
    def __init__(self, sequences):
        self.data = sequences

    def sample_batch(self, batch_size, block_size, device):
        x_batch = []
        y_batch = []
        while len(x_batch) < batch_size:
            seq = random.choice(self.data)
            if len(seq) < block_size + 1:
                continue
            start = random.randint(0, len(seq) - block_size - 1)
            chunk = seq[start:start + block_size + 1]
            x_batch.append(torch.tensor(chunk[:-1], dtype=torch.long))
            y_batch.append(torch.tensor(chunk[1:], dtype=torch.long))
        x = torch.stack(x_batch).to(device)
        y = torch.stack(y_batch).to(device)
        return x, y

def get_scheduler(optimizer, total_steps, warmup_steps):
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=max(1, warmup_steps))
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

def steps_per_epoch(total_tokens, batch_size, block_size):
    tokens_per_step = batch_size * block_size
    return max(1, math.ceil(total_tokens / tokens_per_step))

def train_one_epoch(model, dataset, optimizer, scheduler, device, steps, batch_size, block_size, log_interval=100):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running = 0.0
    recorded = []
    start = time.time()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    for step in range(1, steps + 1):
        x, y = dataset.sample_batch(batch_size, block_size, device)
        out = model(x)
        loss = criterion(out.view(-1, out.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item()
        if step % log_interval == 0 or step == steps:
            interval = min(log_interval, step)
            avg = running / interval
            recorded.append(avg)
            print(f"Step {step}/{steps}, avg_loss={avg:.4f}, lr={scheduler.get_last_lr()[0]:.6g}")
            running = 0.0
    elapsed = time.time() - start
    throughput = steps * batch_size * block_size / elapsed
    peak = None
    if device == 'cuda':
        peak = torch.cuda.max_memory_allocated() / 1024**3
    return recorded, elapsed, throughput, peak

def evaluate(model, dataset, device, steps=100, batch_size=8, block_size=512):
    model.eval()
    crit = nn.CrossEntropyLoss()
    s = 0.0
    with torch.no_grad():
        for _ in range(steps):
            x, y = dataset.sample_batch(batch_size, block_size, device)
            out = model(x)
            loss = crit(out.view(-1, out.size(-1)), y.view(-1))
            s += loss.item()
    return s / steps

def fit_scaling_law(params, losses):
    params = np.asarray(params)
    losses = np.asarray(losses)
    best = None
    for c in np.linspace(0.0, losses.min() * 0.9, 200):
        adjusted = losses - c
        if np.any(adjusted <= 0):
            continue
        lp = np.log(params)
        la = np.log(adjusted)
        coeffs = np.polyfit(lp, la, 1)
        alpha = -coeffs[0]
        a = math.exp(coeffs[1])
        pred = a * params ** (-alpha) + c
        rss = np.sum((pred - losses) ** 2)
        if best is None or rss < best['rss']:
            best = {'a': a, 'alpha': alpha, 'c': c, 'rss': rss}
    if best is None:
        raise RuntimeError('Fit failed')
    return best

def plot_comparison(sp_results, mup_results, fit_sp, fit_mup, outdir):
    plt.figure(figsize=(8, 6))
    sp_params = [r['params'] for r in sp_results]
    sp_losses = [r['val_loss'] for r in sp_results]
    mup_params = [r['params'] for r in mup_results]
    mup_losses = [r['val_loss'] for r in mup_results]
    plt.plot(sp_params, sp_losses, 'o-', label='SP')
    plt.plot(mup_params, mup_losses, 's-', label='µP')
    x_line = np.logspace(math.log10(min(sp_params + mup_params)), math.log10(max(sp_params + mup_params)), 200)
    y_sp = fit_sp['a'] * x_line ** (-fit_sp['alpha']) + fit_sp['c']
    y_mup = fit_mup['a'] * x_line ** (-fit_mup['alpha']) + fit_mup['c']
    plt.plot(x_line, y_sp, '--', color='C0', alpha=0.6, label=f'SP fit α={fit_sp["alpha"]:.3f}')
    plt.plot(x_line, y_mup, '--', color='C1', alpha=0.6, label=f'µP fit α={fit_mup["alpha"]:.3f}')
    plt.xscale('log')
    plt.xlabel('Parameters (M)')
    plt.ylabel('Validation Loss')
    plt.title('SP vs µP Scaling Laws')
    plt.grid(True, which='both', ls='--', alpha=0.4)
    plt.legend()
    path = os.path.join(outdir, 'sp_vs_mup_scaling.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    return path

def apply_mup(model, base_model):
    if not MUP_AVAILABLE:
        raise ImportError("Install mup: pip install mup")
    mup.set_base_shapes(model, base_model)
    # Initialize weights as per µP
    for module in model.modules():
        if isinstance(module, nn.Linear):
            mup.init.kaiming_normal_(module.weight, mode='fan_in')
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0)
        elif isinstance(module, MuReadout):
            # MuReadout handles its own init
            pass
    return model

def extrapolate(fit, max_trained_param):
    target = max_trained_param * 10.0
    pred = fit['a'] * target ** (-fit['alpha']) + fit['c']
    uncertainty = math.sqrt(fit['rss']) / max(1.0, len(fit) - 2)  # rough
    return target, pred, uncertainty

def list_checkpoints(output_dir):
    """List all available checkpoints."""
    checkpoints = {
        'lr_sweep': os.path.exists(os.path.join(output_dir, 'lr_results.pt')),
        'model_training': os.path.exists(os.path.join(output_dir, 'mup_results.pt')),
        'scaling_fit': os.path.exists(os.path.join(output_dir, 'fit_mup.pt')),
        'plots': os.path.exists(os.path.join(output_dir, 'sp_vs_mup_scaling.png')),
    }
    return checkpoints

def clear_checkpoints(output_dir, stages=['lr_sweep', 'model_training', 'scaling_fit']):
    """Clear specific checkpoints to restart from scratch or specific stage.
    stages: list of ['lr_sweep', 'model_training', 'scaling_fit']
    """
    files_to_remove = {
        'lr_sweep': ['lr_results.pt', 'best_lr.txt'],
        'model_training': ['mup_results.pt'],
        'scaling_fit': ['fit_mup.pt', 'part3_results.pt'],
    }
    for stage in stages:
        for filename in files_to_remove.get(stage, []):
            path = os.path.join(output_dir, filename)
            if os.path.exists(path):
                os.remove(path)
                print(f"Removed {path}")
    print(f"Cleared checkpoints for stages: {stages}")

# Load SP results from Part 2
sp_results = []
for name in ['tiny', 'small', 'medium', 'large', 'xl']:
    path = os.path.join(PART2_DIR, f"{name}.pt")
    if os.path.exists(path):
        sp_results.append(torch.load(path))
    else:
        print(f"Warning: {path} not found")

if not sp_results:
    raise FileNotFoundError("No SP results from Part 2 found. Run Part 2 first.")

fit_sp = fit_scaling_law([r['params'] for r in sp_results], [r['val_loss'] for r in sp_results])

# Load data
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
train_tokens = torch.load(TRAIN_TOKENS_PATH)
val_tokens = torch.load(VAL_TOKENS_PATH)
total_train = sum(len(s) for s in train_tokens)

vocab_size = tokenizer.get_vocab_size()
pad_id = None

block_size = 512
tokens_per_batch = 4096
batch_size = max(32, tokens_per_batch // block_size)
epoch_steps = steps_per_epoch(total_train, batch_size, block_size)
warmup_steps = max(1, epoch_steps // 5)

train_ds = TokenDataset(train_tokens)
val_ds = TokenDataset(val_tokens)

model_sizes = [
    ModelConfig('tiny',   128, 6, 4, 512),   # 128/4 = 32
    ModelConfig('small',  192, 6, 6, 768),   # 192/6 = 32
    ModelConfig('medium', 384, 6, 12, 1536), # 384/12 = 32
    ModelConfig('large',  512, 6, 16, 2048), # 512/16 = 32
    ModelConfig('xl',     768, 6, 24, 3072), # 768/24 = 32
]

# Checkpointing paths
lr_results_path = os.path.join(OUTPUT_DIR, 'lr_results.pt')
mup_results_path = os.path.join(OUTPUT_DIR, 'mup_results.pt')
fit_mup_path = os.path.join(OUTPUT_DIR, 'fit_mup.pt')
best_lr_path = os.path.join(OUTPUT_DIR, 'best_lr.txt')
checkpoint_meta_path = os.path.join(OUTPUT_DIR, 'checkpoint_meta.pt')



PART 3: µP SCALING AND EXTRAPOLATION
Output directory: part3_mup_results
Note: This script uses checkpoint/resume logic.
  If interrupted, run again to resume from the last completed stage.
  Stages: 1) LR sweep 2) Train models 3) Fit scaling 4) Plot & analyze
Resuming from checkpoint: Loading LR sweep results...
Loaded best µP LR: 0.001
Resuming from checkpoint: Loading µP training results...
Skipping tiny, already trained
Skipping small, already trained
Skipping medium, already trained
Skipping large, already trained
Loaded 4 µP model results
Resuming from checkpoint: Loading µP scaling law fit...
Loaded µP fit: α=0.4942, a=0.364900, c=0.624629

Generating comparison plot...
Plot saved to part3_mup_results/sp_vs_mup_scaling.png

Extrapolating to 10x model size...
Extrapolated at 250.66M params: loss=0.648427 ± 0.002723

PART 3 SUMMARY
SP scaling exponent α: -0.3135
µP scaling exponent α: 0.4942
SP results: 5 models, 2.3M - 94.3M params
µP results: 4 models, 2.7M - 25.1M params
µP ext

In [7]:
# LR sweep for µP tiny
if not MUP_AVAILABLE:
    print("mup not available, install pip install mup")
else:
    # ============ STAGE 1: LR SWEEP ============
    if os.path.exists(lr_results_path) and os.path.exists(best_lr_path):
        print("Resuming from checkpoint: Loading LR sweep results...")
        lr_results = torch.load(lr_results_path,weights_only=False)
        with open(best_lr_path, 'r') as f:
            best_lr = float(f.read().strip())
        print(f"Loaded best µP LR: {best_lr}")
    else:
        print("Running µP LR sweep...")
        tiny_cfg = model_sizes[0]
        base_model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
        lr_candidates = np.logspace(-5, -3, 7)
        best_lr = None
        best_val = float('inf')
        lr_results = []
        for i, lr in enumerate(lr_candidates):
            print(f"  LR sweep: {i+1}/{len(lr_candidates)}, lr={lr:.6f}")
            model = DecoderOnlyTransformer(vocab_size, tiny_cfg.d_model, tiny_cfg.n_layers, tiny_cfg.n_heads, tiny_cfg.d_ff, mup=True)
            model = apply_mup(model, base_model)
            model = model.to(device)
            optimizer = optim.AdamW(model.parameters(), lr=lr)
            scheduler = get_scheduler(optimizer, max(1, min(epoch_steps, 800)), max(1, warmup_steps//4))
            train_one_epoch(model, train_ds, optimizer, scheduler, device, min(epoch_steps, 800), batch_size, block_size)
            val_loss = evaluate(model, val_ds, device)
            lr_results.append({'lr': lr, 'val_loss': val_loss})
            if val_loss < best_val:
                best_val = val_loss
                best_lr = lr
            print(f"    val_loss={val_loss:.4f}, best_lr={best_lr:.6f}")
        
        # Checkpoint LR sweep
        torch.save(lr_results, lr_results_path)
        with open(best_lr_path, 'w') as f:
            f.write(str(best_lr))
        print(f"Checkpointed LR sweep. Best µP LR: {best_lr}")

    # ============ STAGE 2: TRAIN ALL µP MODELS ============
    # ============ STAGE 2: TRAIN ALL µP MODELS ============
    if os.path.exists(mup_results_path):
        print("Resuming from checkpoint: Loading µP training results...")
        mup_results = torch.load(mup_results_path, weights_only=False)
    else:
        print("No checkpoint found. Starting fresh training...")
        mup_results = []
    
    trained_names = set(r['name'] for r in mup_results)
    print(f"Already trained: {sorted(trained_names)}")
    
    tiny_cfg = model_sizes[0]
    base_model = DecoderOnlyTransformer(
        vocab_size,
        tiny_cfg.d_model,
        tiny_cfg.n_layers,
        tiny_cfg.n_heads,
        tiny_cfg.d_ff,
        mup=True
    )
    
    for i, cfg in enumerate(model_sizes):
        if cfg.name in trained_names:
            print(f"Skipping {cfg.name}, already trained")
            continue
    
        print(f"🚀 Training µP {cfg.name} ({i+1}/{len(model_sizes)})...")
    
        model = DecoderOnlyTransformer(
            vocab_size,
            cfg.d_model,
            cfg.n_layers,
            cfg.n_heads,
            cfg.d_ff,
            mup=True
        )
        model = apply_mup(model, base_model)
        model = model.to(device)
    
        param_count = sum(p.numel() for p in model.parameters()) / 1e6
    
        # safer LR for big models
        lr = best_lr
        if cfg.name in ['large', 'xl']:
            lr = best_lr * 0.5
    
        optimizer = optim.AdamW(model.parameters(), lr=lr)
        scheduler = get_scheduler(optimizer, epoch_steps, warmup_steps)
    
        # avoid OOM for XL
        bs = batch_size if cfg.name != 'xl' else 4
    
        train_losses, elapsed, throughput, peak = train_one_epoch(
            model, train_ds, optimizer, scheduler,
            device, epoch_steps, bs, block_size
        )
    
        val_loss = evaluate(model, val_ds, device)
    
        mup_results.append({
            'name': cfg.name,
            'params': param_count,
            'val_loss': val_loss,
            'train_curve': train_losses
        })
    
        torch.save(mup_results, mup_results_path)
        print(f"✅ Finished {cfg.name}")
    
    print(f"Now have {len(mup_results)} trained models")

    # ============ STAGE 3: FIT SCALING LAW ============
    if os.path.exists(fit_mup_path):
        print("Resuming from checkpoint: Loading µP scaling law fit...")
        fit_mup = torch.load(fit_mup_path,weights_only=False)
        print(f"Loaded µP fit: α={fit_mup['alpha']:.4f}, a={fit_mup['a']:.6f}, c={fit_mup['c']:.6f}")
    else:
        print("Fitting µP scaling law...")
        fit_mup = fit_scaling_law([r['params'] for r in mup_results], [r['val_loss'] for r in mup_results])
        torch.save(fit_mup, fit_mup_path,weights_only=False)
        print(f"Checkpointed µP scaling law: α={fit_mup['alpha']:.4f}, a={fit_mup['a']:.6f}, c={fit_mup['c']:.6f}")

    # ============ STAGE 4: PLOTTING AND ANALYSIS ============
    print("\nGenerating comparison plot...")
    plot_comparison(sp_results, mup_results, fit_sp, fit_mup, OUTPUT_DIR)
    print(f"Plot saved to {OUTPUT_DIR}/sp_vs_mup_scaling.png")

    # Extrapolate
    print("\nExtrapolating to 10x model size...")
    max_param = max(r['params'] for r in mup_results)
    target, pred, unc = extrapolate(fit_mup, max_param)
    print(f"Extrapolated at {target:.2f}M params: loss={pred:.6f} ± {unc:.6f}")

    # Final summary and save
    print("\n" + "="*60)
    print("PART 3 SUMMARY")
    print("="*60)
    print(f"SP scaling exponent α: {fit_sp['alpha']:.4f}")
    print(f"µP scaling exponent α: {fit_mup['alpha']:.4f}")
    print(f"SP results: {len(sp_results)} models, {min(r['params'] for r in sp_results):.1f}M - {max(r['params'] for r in sp_results):.1f}M params")
    print(f"µP results: {len(mup_results)} models, {min(r['params'] for r in mup_results):.1f}M - {max(r['params'] for r in mup_results):.1f}M params")
    print(f"µP extrapolated loss at {target:.2f}M params: {pred:.6f}")
    print("="*60)

    # Save all results
    final_results = {
        'sp_results': sp_results, 
        'mup_results': mup_results, 
        'fit_sp': fit_sp, 
        'fit_mup': fit_mup,
        'best_lr': best_lr,
        'lr_results': lr_results,
        'extrapolation': {'target': target, 'pred': pred, 'unc': unc}
    }
    torch.save(final_results, os.path.join(OUTPUT_DIR, 'part3_results.pt'))
    print(f"\nAll results saved to {OUTPUT_DIR}/part3_results.pt")

Resuming from checkpoint: Loading LR sweep results...
Loaded best µP LR: 0.001
Resuming from checkpoint: Loading µP training results...
Already trained: ['large', 'medium', 'small', 'tiny']
Skipping tiny, already trained
Skipping small, already trained
Skipping medium, already trained
Skipping large, already trained
🚀 Training µP xl (5/5)...
Step 100/7348, avg_loss=6.4880, lr=3.40372e-05
Step 200/7348, avg_loss=4.8963, lr=6.8074e-05
Step 300/7348, avg_loss=3.8773, lr=0.000102111
Step 400/7348, avg_loss=2.6470, lr=0.000136147
Step 500/7348, avg_loss=1.9737, lr=0.000170184
Step 600/7348, avg_loss=1.8475, lr=0.000204221
Step 700/7348, avg_loss=1.7616, lr=0.000238258
Step 800/7348, avg_loss=1.7303, lr=0.000272294
Step 900/7348, avg_loss=1.7080, lr=0.000306331
Step 1000/7348, avg_loss=1.6561, lr=0.000340368
Step 1100/7348, avg_loss=1.6266, lr=0.000374404
Step 1200/7348, avg_loss=1.6273, lr=0.000408441
Step 1300/7348, avg_loss=1.6337, lr=0.000442478
Step 1400/7348, avg_loss=1.6102, lr=0.0004

/home/av4008/.local/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Step 1500/7348, avg_loss=1.6081, lr=0.000499966
Step 1600/7348, avg_loss=1.5804, lr=0.000499388
Step 1700/7348, avg_loss=1.5543, lr=0.000498098
Step 1800/7348, avg_loss=1.5498, lr=0.000496099
Step 1900/7348, avg_loss=1.4917, lr=0.000493399
Step 2000/7348, avg_loss=1.5474, lr=0.000490003
Step 2100/7348, avg_loss=1.5176, lr=0.000485922
Step 2200/7348, avg_loss=1.4563, lr=0.000481167
Step 2300/7348, avg_loss=1.4638, lr=0.000475753
Step 2400/7348, avg_loss=1.4790, lr=0.000469694
Step 2500/7348, avg_loss=1.4303, lr=0.000463008
Step 2600/7348, avg_loss=1.4737, lr=0.000455714
Step 2700/7348, avg_loss=1.3871, lr=0.000447832
Step 2800/7348, avg_loss=1.3942, lr=0.000439386
Step 2900/7348, avg_loss=1.3739, lr=0.000430399
Step 3000/7348, avg_loss=1.3533, lr=0.000420897
Step 3100/7348, avg_loss=1.3654, lr=0.000410907
Step 3200/7348, avg_loss=1.3625, lr=0.000400458
Step 3300/7348, avg_loss=1.3288, lr=0.000389579
Step 3400/7348, avg_loss=1.3139, lr=0.000378302
Step 3500/7348, avg_loss=1.2885, lr=0.00